In [1]:
import torch
import torch.nn as nn
import numpy as np

In [2]:

# Define the MLP class with two layers
class MLPclass(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=128, output_dim=3):
        super(MLPclass, self).__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim, bias=True)
        self.layer2 = nn.Linear(hidden_dim, output_dim, bias=True)

    def forward(self, x):
        x = self.layer1(x)
        x = torch.relu(x)
        x = self.layer2(x)
        return x

# Load trained weights
W_loaded1 = torch.load('dim=128_layer=0.pth')  # Should contain both layers' weights & biases
W_loaded2 = torch.load('dim=128_layer=1.pth')  # Should contain both layers' weights & biases

# Create an untrained model instance
mlp = MLPclass()

# Extract default initialized weights and biases
W1_default = mlp.layer1.weight  # First layer weights
b1_default = mlp.layer1.bias    # First layer bias
W2_default = mlp.layer2.weight  # Second layer weights
b2_default = mlp.layer2.bias    # Second layer bias

# Extract trained weights and biases (assuming correct keys exist)
W1_loaded = W_loaded1['linear.weight']
b1_loaded = W_loaded1['linear.bias']
W2_loaded = W_loaded2['linear.weight']
b2_loaded = W_loaded2['linear.bias']



/tmp/ipykernel_3859213/964172190.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  W_loaded1 = torch.load('dim=128_layer=0.pth')  # Should contain both layers' weights & 

In [3]:
# Compute mean and variance for Layer 1 weights and bias
mean_W1 = torch.mean(W1_loaded)
var_W1 = torch.var(W1_loaded)

mean_b1 = torch.mean(b1_loaded)
var_b1 = torch.var(b1_loaded)

# Compute mean and variance for Layer 2 weights and bias
mean_W2 = torch.mean(W2_loaded)
var_W2 = torch.var(W2_loaded)

mean_b2 = torch.mean(b2_loaded)
var_b2 = torch.var(b2_loaded)

# Print results
print(f"Layer 1 Weights - Mean: {mean_W1.item()}, Variance: {var_W1.item()}")
print(f"Layer 1 Bias - Mean: {mean_b1.item()}, Variance: {var_b1.item()}")
print(f"Layer 2 Weights - Mean: {mean_W2.item()}, Variance: {var_W2.item()}")
print(f"Layer 2 Bias - Mean: {mean_b2.item()}, Variance: {var_b2.item()}")


Layer 1 Weights - Mean: -0.10490346699953079, Variance: 0.3158491849899292
Layer 1 Bias - Mean: 0.02118445374071598, Variance: 0.24425379931926727
Layer 2 Weights - Mean: -0.00027134394622407854, Variance: 0.3466038107872009
Layer 2 Bias - Mean: 0.09608057886362076, Variance: 0.00849873572587967


In [3]:
# Compute L2 Norm (Euclidean Distance)
def compute_l2_norm(W1, W2):
    return torch.norm(W1 - W2, p=2).item()

# Compute Mean Squared Error (MSE)
def compute_mse(W1, W2):
    return torch.mean((W1 - W2) ** 2).item()

# Compute Cosine Similarity
def compute_cosine_similarity(W1, W2):
    return torch.nn.functional.cosine_similarity(W1.flatten(), W2.flatten(), dim=0).item()

# Compute the difference metrics
distance_W1 = compute_l2_norm(W1_loaded, W1_default)
mse_W1 = compute_mse(W1_loaded, W1_default)
cosine_W1 = compute_cosine_similarity(W1_loaded, W1_default)

distance_b1 = compute_l2_norm(b1_loaded, b1_default)
mse_b1 = compute_mse(b1_loaded, b1_default)
cosine_b1 = compute_cosine_similarity(b1_loaded, b1_default)

distance_W2 = compute_l2_norm(W2_loaded, W2_default)
mse_W2 = compute_mse(W2_loaded, W2_default)
cosine_W2 = compute_cosine_similarity(W2_loaded, W2_default)

distance_b2 = compute_l2_norm(b2_loaded, b2_default)
mse_b2 = compute_mse(b2_loaded, b2_default)
cosine_b2 = compute_cosine_similarity(b2_loaded, b2_default)

# Print the results
print(f"Layer 1 Weights - L2 Distance: {distance_W1}, MSE: {mse_W1}, Cosine Similarity: {cosine_W1}")
print(f"Layer 1 Bias - L2 Distance: {distance_b1}, MSE: {mse_b1}, Cosine Similarity: {cosine_b1}")

print(f"Layer 2 Weights - L2 Distance: {distance_W2}, MSE: {mse_W2}, Cosine Similarity: {cosine_W2}")
print(f"Layer 2 Bias - L2 Distance: {distance_b2}, MSE: {mse_b2}, Cosine Similarity: {cosine_b2}")


Layer 1 Weights - L2 Distance: 13.020328521728516, MSE: 0.44148167967796326, Cosine Similarity: -0.0034921872429549694
Layer 1 Bias - L2 Distance: 7.012534141540527, MSE: 0.38418465852737427, Cosine Similarity: -0.059619370847940445
Layer 2 Weights - L2 Distance: 11.582695960998535, MSE: 0.349371999502182, Cosine Similarity: -0.01827538199722767
Layer 2 Bias - L2 Distance: 0.23912563920021057, MSE: 0.01906035654246807, Cosine Similarity: -0.04527294635772705


In [4]:
# Number of random initializations to sample
N = 100

# Store L2 distances between different random initializations
random_distances_W1 = []
random_distances_b1 = []
random_distances_W2 = []
random_distances_b2 = []

for _ in range(N):
    # Generate a new random initialization
    mlp_random = MLPclass()

    # Extract random weights & biases
    W1_rand = mlp_random.layer1.weight.clone()
    b1_rand = mlp_random.layer1.bias.clone()
    W2_rand = mlp_random.layer2.weight.clone()
    b2_rand = mlp_random.layer2.bias.clone()

    # Compute L2 distances between different random initializations
    random_distances_W1.append(compute_l2_norm(W1_rand, W1_default))
    random_distances_b1.append(compute_l2_norm(b1_rand, b1_default))
    random_distances_W2.append(compute_l2_norm(W2_rand, W2_default))
    random_distances_b2.append(compute_l2_norm(b2_rand, b2_default))

# Convert to numpy arrays for statistical analysis
random_distances_W1 = np.array(random_distances_W1)
random_distances_b1 = np.array(random_distances_b1)
random_distances_W2 = np.array(random_distances_W2)
random_distances_b2 = np.array(random_distances_b2)

# Compute mean and standard deviation of random L2 distances
mu_W1, sigma_W1 = np.mean(random_distances_W1), np.std(random_distances_W1)
mu_b1, sigma_b1 = np.mean(random_distances_b1), np.std(random_distances_b1)
mu_W2, sigma_W2 = np.mean(random_distances_W2), np.std(random_distances_W2)
mu_b2, sigma_b2 = np.mean(random_distances_b2), np.std(random_distances_b2)

# Compute L2 distances between trained and default initialized weights
distance_W1 = compute_l2_norm(W1_loaded, W1_default)
distance_b1 = compute_l2_norm(b1_loaded, b1_default)
distance_W2 = compute_l2_norm(W2_loaded, W2_default)
distance_b2 = compute_l2_norm(b2_loaded, b2_default)

# Compute z-scores to check significance
z_W1 = (distance_W1 - mu_W1) / sigma_W1
z_b1 = (distance_b1 - mu_b1) / sigma_b1
z_W2 = (distance_W2 - mu_W2) / sigma_W2
z_b2 = (distance_b2 - mu_b2) / sigma_b2

# Print the results
print(f"Layer 1 Weights - L2 Distance: {distance_W1}, Mean Random L2: {mu_W1}, Z-score: {z_W1}")
print(f"Layer 1 Bias - L2 Distance: {distance_b1}, Mean Random L2: {mu_b1}, Z-score: {z_b1}")
print(f"Layer 2 Weights - L2 Distance: {distance_W2}, Mean Random L2: {mu_W2}, Z-score: {z_W2}")
print(f"Layer 2 Bias - L2 Distance: {distance_b2}, Mean Random L2: {mu_b2}, Z-score: {z_b2}")

# Interpretation:
alpha = 0.05  # 95% confidence level
threshold = 1.96  # Corresponding Z-score threshold for significance

print("\nSignificance Test:")
print(f"Layer 1 Weights - Significant? {'Yes' if abs(z_W1) > threshold else 'No'}")
print(f"Layer 1 Bias - Significant? {'Yes' if abs(z_b1) > threshold else 'No'}")
print(f"Layer 2 Weights - Significant? {'Yes' if abs(z_W2) > threshold else 'No'}")
print(f"Layer 2 Bias - Significant? {'Yes' if abs(z_b2) > threshold else 'No'}")


Layer 1 Weights - L2 Distance: 13.020328521728516, Mean Random L2: 9.311550598144532, Z-score: 14.134157912206037
Layer 1 Bias - L2 Distance: 7.012534141540527, Mean Random L2: 5.41761061668396, Z-score: 6.0113941643589826
Layer 2 Weights - L2 Distance: 11.582695960998535, Mean Random L2: 1.4097346019744874, Z-score: 274.66681523397324
Layer 2 Bias - L2 Distance: 0.23912563920021057, Mean Random L2: 0.12493470643647014, Z-score: 2.3088478076016377

Significance Test:
Layer 1 Weights - Significant? Yes
Layer 1 Bias - Significant? Yes
Layer 2 Weights - Significant? Yes
Layer 2 Bias - Significant? Yes
